<a href="https://colab.research.google.com/github/mohamadfaisalbashir/grokking-Deep-Learning/blob/main/13_introducing_automatic_optimization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Chapter 13: Introducing Automatic Optimization - Let's Build a Deep Learning Framework**

This notebook summarizes Chapter 13 of *Grokking Deep Learning*:

1. What Is a Deep Learning Framework?
2. Introduction to Tensors
3. Introduction to Automatic Gradient Computation (Autograd)
4. A Quick Checkpoint
5. Tensors That Are Used Multiple Times
6. Upgrading Autograd to Support Multiuse Tensors
7. How Does Addition Backpropagation Work?
8. Adding Support for Negation
9. Adding Support for Additional Functions
10. Using Autograd to Train a Neural Network
11. Adding Automatic Optimization
12. Adding Support for Layer Types
13. How to Learn a Framework
14. Nonlinearity Layers
15. The Embedding Layer
16. The Cross-Entropy Layer
17. The Recurrent Neural Network Layer
18. Summary

Until now, every network in the book was written in plain NumPy, with the backpropagation code derived and typed by hand. This chapter changes the working style: instead of adopting an existing framework, the author **builds a small deep learning framework from scratch**, step by step. It starts from a `Tensor` class that stores numbers, gives it the ability to compute gradients automatically (**autograd**), adds more operations, and then adds an **optimizer**, **layers**, **loss functions**, **nonlinearities**, an **embedding layer**, a **cross-entropy** loss and a **recurrent layer**. In the end, the recurrent network of the previous chapter is retrained with this little framework.

The chapter opens with a remark by Arthur C. Clarke (*2010: Odyssey Two*, 1982): being based on carbon or on silicon makes no fundamental difference, and each of us should be treated with appropriate respect.

---
## **1. What Is a Deep Learning Framework?**

**Good tools reduce mistakes, speed up development, and make programs run faster.**

### **1.1 Why Move to a Framework Now?**

Anyone who has followed deep learning for a while has met the major frameworks, such as PyTorch, TensorFlow, Theano (recently deprecated at the time of writing), Keras, Lasagne, and DyNet. They have evolved quickly, and although all of them are free and open source, there is a friendly spirit of competition around each one.

So far the book has avoided them on purpose: it is important to know what happens under the hood, which is why every algorithm has been implemented from scratch in NumPy. The networks coming next, **long short-term memory networks (LSTMs)**, are so complex that their NumPy implementation is hard to read, use, or debug (the text jokes that gradients fly everywhere). Reducing exactly this kind of code complexity is what frameworks were invented for. They also matter for speed: training on a **GPU** can be 10 to 100 times faster, and a framework makes that practical while keeping the code simple. For these reasons nearly all researchers use one, and a solid understanding of a framework is essential for anybody who wants to use or research deep learning.

### **1.2 Building a Light Framework Instead of Using a Famous One**

Jumping straight into a well-known framework would hide what complex models such as LSTMs actually do. Therefore, the reader builds a **light framework that follows the latest trends in framework design**. Afterwards there is no doubt about what a framework does for complicated architectures, and the move to a real framework is smooth, because its API and its inner workings are already familiar. The author found the exercise valuable himself, especially for debugging troublesome models.

Abstractly, a framework removes code that would otherwise be repeated many times. Concretely, its most valuable features are **automatic backpropagation** and **automatic optimization**: the user writes only the forward propagation code, and the framework takes care of backpropagation and weight updates. Many frameworks also simplify forward propagation itself with high-level interfaces for common layers and loss functions.

### **1.3 How the Code Is Organized in This Notebook**

The chapter grows **one `Tensor` class** over many pages, and the book usually prints only the new or changed methods of the class (and the new `if` branch to be placed inside `backward()`). To keep every cell runnable in sequence, this notebook gives, whenever the class changes, a **complete cell with the new version of the class**. These cells are assembled only from the lines printed in the book, and each new cell replaces the previous definition of `Tensor`. The labels "v1" to "v8" below are this notebook's own names for the versions.

| Version | Section | What is new |
|---|---|---|
| v1 | 2 | `data`, addition, `__repr__`, `__str__` |
| v2 | 3 | `creators`, `creation_op`, `grad`, a first `backward` (addition only) |
| v3 | 6 | `autograd` flag, `id`, `children` counters, gradient accumulation |
| v4 | 8 | negation (`__neg__`) |
| v5 | 9 | `__sub__`, `__mul__`, `sum`, `expand`, `transpose`, `mm` |
| v6 | 13 and 14 | default gradient of ones in `backward`, `sigmoid`, `tanh` |
| v7 | 15 | `index_select` |
| v8 | 16 | `cross_entropy` |

The book does not say where new methods must be placed inside the class. In the assembled cells, the new forward methods are inserted after `__add__` (before `__repr__`), and each new `backward` branch is placed after the previous one, inside the block that is executed once the gradients of all children have arrived.

Two general remarks apply to the whole notebook. First, several listings of the book start with `import numpy` and then call `np.random.seed(0)`, which relies on `np` having been imported earlier (the first cell does `import numpy as np`), so run the cells in order. Second, the very last part of the chapter needs a data file that is not part of this notebook (see section 17).

---
## **2. Introduction to Tensors**

**A tensor generalizes vectors and matrices to any number of dimensions.**

Up to this point, vectors and matrices have been the basic data structures. A matrix is a list of vectors, and a vector is a list of scalars (single numbers). A **tensor** is the abstract version of such nested lists of numbers: a vector is a one-dimensional tensor, a matrix is a two-dimensional tensor, and higher dimensions are called *n*-dimensional tensors. The first brick of the new framework is therefore a basic type named `Tensor`.

The first version (v1) keeps all numbers in a NumPy array, `self.data`, and supports exactly one tensor operation, addition. The methods `__repr__` and `__str__` only make printing convenient. Adding further operations is simple: one writes more methods on the class.

In [1]:
import numpy as np

class Tensor (object):

    def __init__(self, data):
        self.data = np.array(data)

    def __add__(self, other):
        return Tensor(self.data + other.data)

    def __repr__(self):
        return str(self.data.__repr__())

    def __str__(self):
        return str(self.data.__str__())

x = Tensor([1,2,3,4,5])
print(x)

y = x + x
print(y)

[1 2 3 4 5]
[ 2  4  6  8 10]


Running the cell prints `[1 2 3 4 5]` for `x` and `[ 2  4  6  8 10]` for `y = x + x`.

---
## **3. Introduction to Automatic Gradient Computation (Autograd)**

**Backpropagation, which used to be done by hand, can be made automatic.**

Since Chapter 4 the reader has computed derivatives by hand for every network: first the gradient at the network's output, then, moving backward, the derivative at the next-to-last component, and so on until every weight has a gradient. This logic can be built into the tensor object itself.

### **3.1 The First Version of Autograd**

Version v2 adds three attributes to each tensor and one method:

- **`creators`**: a list of the tensors that were used to create this tensor (default `None`);
- **`creation_op`**: the operation that was applied to those creators (here the string `"add"`);
- **`grad`**: the gradient that will be stored in the tensor;
- **`backward(grad)`**: stores the gradient and, if the tensor was created by an addition, passes the same gradient on to both creators.

The `__add__` method now records `creators=[self,other]` and `creation_op="add"` in the tensor it returns.

In [2]:
import numpy as np

class Tensor (object):

    def __init__(self, data, creators=None, creation_op=None):
        self.data = np.array(data)
        self.creation_op = creation_op
        self.creators = creators
        self.grad = None

    def backward(self, grad):
        self.grad = grad

        if(self.creation_op == "add"):
            self.creators[0].backward(grad)
            self.creators[1].backward(grad)

    def __add__(self, other):
        return Tensor(self.data + other.data,
                      creators=[self,other],
                      creation_op="add")

    def __repr__(self):
        return str(self.data.__repr__())

    def __str__(self):
        return str(self.data.__str__())

x = Tensor([1,2,3,4,5])
y = Tensor([2,2,2,2,2])

z = x + y
z.backward(Tensor(np.array([1,1,1,1,1])))

### **3.2 The Computation Graph**

Writing `z = x + y` now builds a **computation graph** with three nodes (`x`, `y`, and `z`) and two edges (from `z` to `x` and from `z` to `y`), each edge labeled with the creation operation `add`. This graph is what allows the gradients to be backpropagated recursively. The graph is created automatically whenever mathematical operations are performed; if `z` were used in further operations, the new tensors would point back to `z` and the graph would keep growing.

> The computation graph created by `z = x + y`: three nodes and two edges, each labeled with the creation operation `add`.
>
> <img src="pictures/chapter 13/picture 13.1.png" alt="Figure 13.1" width="600"/>

The second new concept is the use of this graph to compute gradients. Calling `z.backward(...)` places a vector of gradients (here a vector of ones) on `z` and sends it to its parents. As learned in Chapter 4, backpropagating through an addition means applying an addition again; because only one gradient has to be added into `x` and into `y`, the gradient of `z` is simply **copied** onto both of them. The next cell inspects the result.

In [3]:
print(x.grad)
print(y.grad)
print(z.creators)
print(z.creation_op)

[1 1 1 1 1]
[1 1 1 1 1]
[array([1, 2, 3, 4, 5]), array([2, 2, 2, 2, 2])]
add


The gradients of `x` and `y` are both `[1 1 1 1 1]`, `z.creators` holds the two arrays that were added, and `z.creation_op` is `add`.

### **3.3 Recursive Backpropagation**

The most elegant aspect of this design is that it works recursively: each tensor calls `.backward()` on all of its `self.creators`. In the next cell, `e = a + b`, `f = c + d`, and `g = e + f`; calling `backward` on `g` sends the gradient all the way down to `a`, whose gradient is again `[1 1 1 1 1]`.

In [4]:
a = Tensor([1,2,3,4,5])
b = Tensor([2,2,2,2,2])
c = Tensor([5,4,3,2,1])
d = Tensor([-1,-2,-3,-4,-5])
e = a + b
f = c + d
g = e + f
g.backward(Tensor(np.array([1,1,1,1,1])))
print(a.grad)

[1 1 1 1 1]


---
## **4. A Quick Checkpoint**

**Nothing in the new `Tensor` is conceptually new; it repackages lessons already learned.**

Even if it feels like a stretch to think of gradients flowing over a graph structure, this is not new. In the previous chapter on RNNs, the network was forward propagated in one direction and then backpropagated across a (virtual) graph of activations. The nodes and edges were just not stored in an explicit data structure: there was a list of layers (dictionaries), and the correct order of forward and backward operations was coded by hand. Now the work goes into a convenient interface that backpropagates **recursively**, so that complicated backpropagation code does not have to be written by hand.

### **4.1 Dynamic versus Static Computation Graphs**

A graph that is built on the fly during forward propagation is called a **dynamic computation graph**. This kind of autograd is found in newer frameworks such as DyNet and PyTorch. Older frameworks such as Theano and TensorFlow use a **static computation graph**, which is specified before forward propagation even begins.

- Dynamic graphs are generally easier to write and experiment with.
- Static graphs have faster runtimes, thanks to optimizations under the hood.

The two families are moving toward each other: dynamic graphs can be compiled into static ones for speed, and static graphs can be built dynamically for easier experimentation, so in the long run one is likely to meet both. The essential difference is whether forward propagation happens *during* graph construction or *after* the graph has been defined. This book sticks with the dynamic approach.

### **4.2 The Point of the Chapter**

The chapter prepares the reader for deep learning in practice, where roughly 10% (or less) of the time goes into thinking up a new idea and 90% into getting a framework to behave. Debugging frameworks can be very hard because most bugs do not raise an error or print a stack trace; they hide in the code and keep the network from training as it should, even though it may appear to train somewhat. The reader is therefore encouraged to study the chapter closely, which pays off, for instance, when chasing an optimization bug at 2:00 a.m.

---
## **5. Tensors That Are Used Multiple Times**

**The basic autograd has an annoying bug when a tensor is used more than once.**

Version v2 can backpropagate into a variable only once. During forward propagation, however, the same tensor is often used several times (the weights of a network are an example), and then several parts of the graph backpropagate gradients into it. The current code then computes the wrong gradient for such a variable, which is the parent of multiple children. The next cell (still using v2) shows the problem: `b` is used in both `d = a + b` and `e = b + c`, so its gradient should be the **sum of two derivatives**, `[2,2,2,2,2]`. The comparison prints `False` everywhere.

In [5]:
a = Tensor([1,2,3,4,5])
b = Tensor([2,2,2,2,2])
c = Tensor([5,4,3,2,1])

d = a + b
e = b + c
f = d + e
f.backward(Tensor(np.array([1,1,1,1,1])))

print(b.grad.data == np.array([2,2,2,2,2]))

[False False False False False]


> The graph of `d = a + b`, `e = b + c`, `f = d + e`: two edges point into `b`, so its gradient must be the sum of the gradients arriving from `d` and from `e`.
>
> <img src="pictures/chapter 13/picture 13.2.png" alt="Figure 13.2" width="600"/>

The reason is that the current implementation simply **overwrites** the stored gradient: first `d` writes its gradient into `b`, and then `e` overwrites it. The way gradients are written has to change.

---
## **6. Upgrading Autograd to Support Multiuse Tensors**

**One new method and three updated ones make gradients accumulate correctly.**

The update (v3) adds two features. First, gradients are **accumulated**, so a variable that is used more than once receives gradients from all of its children. Second, a counter prevents a tensor from backpropagating from the same child twice (which raises an exception). The changes, which concern `__init__`, `backward`, and `__add__` plus one new method, are the following:

- **`autograd`** (default `False`): a flag that tells whether the tensor takes part in automatic differentiation.
- **`id`**: an identifier of the tensor; if none is given, a random integer between 0 and 100000 is drawn.
- **`children`**: a dictionary that counts, for each child (identified by its `id`), how many times that child used this tensor. It is filled in the constructor of the child.
- **`all_children_grads_accounted_for()`**: returns `True` only if every counter in `children` is back at zero, i.e., if the tensor has received the gradient of each of its children.
- **`backward(grad=None, grad_origin=None)`**: it runs only if `autograd` is `True`. If the call comes from a child (`grad_origin`), it first checks that this child's counter is not already zero (otherwise it raises `"cannot backprop more than once"`) and then decrements it. The incoming gradient is stored if there is none yet and **added** to the stored one otherwise. Backpropagation to the creators begins only when all children have delivered their gradients, or when `grad_origin` is `None` (the tensor where the backward pass starts).
- **`__add__`**: the returned tensor is part of the graph (with `autograd=True`) only if **both** operands have `autograd` turned on; otherwise a plain tensor is returned.

Every tensor that must receive a gradient is therefore created with `autograd=True`. The cell below contains the complete class (v3) followed by the test of the previous section; the comparison now prints `True` five times.

In [6]:
import numpy as np
class Tensor (object):

    def __init__(self,data,
                 autograd=False,
                 creators=None,
                 creation_op=None,
                 id=None):

        self.data = np.array(data)
        self.creators = creators
        self.creation_op = creation_op
        self.grad = None
        self.autograd = autograd
        self.children = {}
        if(id is None):
            id = np.random.randint(0,100000)
        self.id = id

        if(creators is not None):
            for c in creators:
                if(self.id not in c.children):
                    c.children[self.id] = 1
                else:
                    c.children[self.id] += 1

    def all_children_grads_accounted_for(self):
        for id,cnt in self.children.items():
            if(cnt != 0):
                return False
        return True

    def backward(self,grad=None, grad_origin=None):
        if(self.autograd):

            if(grad_origin is not None):
                if(self.children[grad_origin.id] == 0):
                    raise Exception("cannot backprop more than once")
                else:
                    self.children[grad_origin.id] -= 1

            if(self.grad is None):
                self.grad = grad
            else:
                self.grad += grad

            if(self.creators is not None and
               (self.all_children_grads_accounted_for() or
                grad_origin is None)):

                if(self.creation_op == "add"):
                    self.creators[0].backward(self.grad, self)
                    self.creators[1].backward(self.grad, self)

    def __add__(self, other):
        if(self.autograd and other.autograd):
            return Tensor(self.data + other.data,
                          autograd=True,
                          creators=[self,other],
                          creation_op="add")
        return Tensor(self.data + other.data)

    def __repr__(self):
        return str(self.data.__repr__())

    def __str__(self):
        return str(self.data.__str__())

a = Tensor([1,2,3,4,5], autograd=True)
b = Tensor([2,2,2,2,2], autograd=True)
c = Tensor([5,4,3,2,1], autograd=True)

d = a + b
e = b + c
f = d + e

f.backward(Tensor(np.array([1,1,1,1,1])))

print(b.grad.data == np.array([2,2,2,2,2]))

[ True  True  True  True  True]


The wait-until-all-children-have-reported logic exists because some variables get their gradient from several parents, and each variable has to postpone calling `.backward()` on its own parents until its gradient is final. None of this is new from the point of view of deep learning theory; these are engineering challenges that frameworks face, and similar ones appear when debugging networks in a standard framework. The reader is invited to play with the code, to delete parts of it and watch it break in different ways, and to call backward twice. (The book writes `.backprop()` here, but the method of the class is called `.backward()`.)

---
## **7. How Does Addition Backpropagation Work?**

**Studying how addition is handled shows how to support any other function.**

The framework has now reached the point where support for an arbitrary operation can be added by writing the function in the `Tensor` class and its derivative in the `.backward()` method. For addition, these are the two places involved. The forward method:

```python
def __add__(self, other):
    if(self.autograd and other.autograd):
        return Tensor(self.data + other.data,
                      autograd=True,
                      creators=[self,other],
                      creation_op="add")
    return Tensor(self.data + other.data)
```

and the gradient propagation inside `.backward()`:

```python
if(self.creation_op == "add"):
    self.creators[0].backward(self.grad, self)
    self.creators[1].backward(self.grad, self)
```

Addition is not handled anywhere else, because the generic backpropagation logic is abstracted away. The branch calls `.backward()` twice, once for each variable that took part in the addition, so by default the gradient is sent into every variable of the graph. It is skipped only for variables whose `autograd` is `False`, which is checked at the top of `.backward()`:

```python
def backward(self,grad=None, grad_origin=None):
    if(self.autograd):

        if(grad_origin is not None):
            if(self.children[grad_origin.id] == 0):
                raise Exception("cannot backprop more than once")

                    ...
```

Hence, even though the addition branch pushes the gradient into all contributing variables, the call has no effect unless `.autograd` is `True` for the receiving variable (`self.creators[0]` or `self.creators[1]`). Likewise, the first line of `__add__` shows that the new tensor (the one that will later call `.backward()`) has `autograd == True` only if both operands do. These excerpts come from the class v3 above, so there is no new code to run in this section.

---
## **8. Adding Support for Negation**

**Negation is obtained by adapting the code for addition.**

Now that addition works, one can copy its code and make a few modifications to obtain negation. The method `__neg__` takes no `other` parameter, builds the tensor from `self.data * -1`, has the single creator `[self]`, and uses the creation operation `"neg"`. In `backward`, the new branch is `if(self.creation_op == "neg")`: it calls `.backward()` **only once**, with the negated gradient `self.grad.__neg__()`, because the operation has a single creator. (If the reader wonders why these are the right gradients, Chapters 4, 5, and 6 explain them.)

Cell v4 below is the complete class including these two additions; it replaces the previous `Tensor`. In the test that follows, `-b` is used in two places, so `b` receives two gradients of `-1` and the result should be `[-2,-2,-2,-2,-2]`.

In [7]:
import numpy as np
class Tensor (object):

    def __init__(self,data,
                 autograd=False,
                 creators=None,
                 creation_op=None,
                 id=None):

        self.data = np.array(data)
        self.creators = creators
        self.creation_op = creation_op
        self.grad = None
        self.autograd = autograd
        self.children = {}
        if(id is None):
            id = np.random.randint(0,100000)
        self.id = id

        if(creators is not None):
            for c in creators:
                if(self.id not in c.children):
                    c.children[self.id] = 1
                else:
                    c.children[self.id] += 1

    def all_children_grads_accounted_for(self):
        for id,cnt in self.children.items():
            if(cnt != 0):
                return False
        return True

    def backward(self,grad=None, grad_origin=None):
        if(self.autograd):

            if(grad_origin is not None):
                if(self.children[grad_origin.id] == 0):
                    raise Exception("cannot backprop more than once")
                else:
                    self.children[grad_origin.id] -= 1

            if(self.grad is None):
                self.grad = grad
            else:
                self.grad += grad

            if(self.creators is not None and
               (self.all_children_grads_accounted_for() or
                grad_origin is None)):

                if(self.creation_op == "add"):
                    self.creators[0].backward(self.grad, self)
                    self.creators[1].backward(self.grad, self)

                if(self.creation_op == "neg"):
                    self.creators[0].backward(self.grad.__neg__())

    def __add__(self, other):
        if(self.autograd and other.autograd):
            return Tensor(self.data + other.data,
                          autograd=True,
                          creators=[self,other],
                          creation_op="add")
        return Tensor(self.data + other.data)

    def __neg__(self):
        if(self.autograd):
            return Tensor(self.data * -1,
                          autograd=True,
                          creators=[self],
                          creation_op="neg")
        return Tensor(self.data * -1)

    def __repr__(self):
        return str(self.data.__repr__())

    def __str__(self):
        return str(self.data.__str__())

In [8]:
a = Tensor([1,2,3,4,5], autograd=True)
b = Tensor([2,2,2,2,2], autograd=True)
c = Tensor([5,4,3,2,1], autograd=True)

d = a + (-b)
e = (-b) + c
f = d + e

f.backward(Tensor(np.array([1,1,1,1,1])))

print(b.grad.data == np.array([-2,-2,-2,-2,-2]))

[ True  True  True  True  True]


The comparison prints `True` for all five entries. When `-b` is used in the forward pass instead of `b`, the backpropagated gradients have their sign flipped as well, and nothing in the general backpropagation system had to be changed. New functions can be created whenever they are needed.

---
## **9. Adding Support for Additional Functions**

**Subtraction, multiplication, sum, expand, transpose, and matrix multiplication follow the same pattern.**

With the ideas used for addition and negation, the book adds the forward and backward logic of six more functions: `__sub__`, `__mul__`, `sum`, `expand`, `transpose`, and `mm` (matrix multiplication). The book prints their forward methods first and the corresponding `backward` branches afterwards; the cell below (v5) joins both into one complete class, replacing the previous one. In the branch for subtraction the book prints `self.creators[1].backward(, self)`, which is a syntax error because the argument is missing; the obvious intention is to pass the negated gradient `new`.

*Note on a repair:* the book prints `self.creators[1].backward(, self)`; this notebook uses `self.creators[1].backward(new, self)`, because the variable `new` was just assigned the negated gradient and the empty argument cannot be parsed.

In [9]:
import numpy as np
class Tensor (object):

    def __init__(self,data,
                 autograd=False,
                 creators=None,
                 creation_op=None,
                 id=None):

        self.data = np.array(data)
        self.creators = creators
        self.creation_op = creation_op
        self.grad = None
        self.autograd = autograd
        self.children = {}
        if(id is None):
            id = np.random.randint(0,100000)
        self.id = id

        if(creators is not None):
            for c in creators:
                if(self.id not in c.children):
                    c.children[self.id] = 1
                else:
                    c.children[self.id] += 1

    def all_children_grads_accounted_for(self):
        for id,cnt in self.children.items():
            if(cnt != 0):
                return False
        return True

    def backward(self,grad=None, grad_origin=None):
        if(self.autograd):

            if(grad_origin is not None):
                if(self.children[grad_origin.id] == 0):
                    raise Exception("cannot backprop more than once")
                else:
                    self.children[grad_origin.id] -= 1

            if(self.grad is None):
                self.grad = grad
            else:
                self.grad += grad

            if(self.creators is not None and
               (self.all_children_grads_accounted_for() or
                grad_origin is None)):

                if(self.creation_op == "add"):
                    self.creators[0].backward(self.grad, self)
                    self.creators[1].backward(self.grad, self)

                if(self.creation_op == "neg"):
                    self.creators[0].backward(self.grad.__neg__())

                if(self.creation_op == "sub"):
                    new = Tensor(self.grad.data)
                    self.creators[0].backward(new, self)
                    new = Tensor(self.grad.__neg__().data)
                    self.creators[1].backward(new, self)

                if(self.creation_op == "mul"):
                    new = self.grad * self.creators[1]
                    self.creators[0].backward(new , self)
                    new = self.grad * self.creators[0]
                    self.creators[1].backward(new, self)

                if(self.creation_op == "mm"):
                    act = self.creators[0]
                    weights = self.creators[1]
                    new = self.grad.mm(weights.transpose())
                    act.backward(new)
                    new = self.grad.transpose().mm(act).transpose()
                    weights.backward(new)

                if(self.creation_op == "transpose"):
                    self.creators[0].backward(self.grad.transpose())

                if("sum" in self.creation_op):
                    dim = int(self.creation_op.split("_")[1])
                    ds = self.creators[0].data.shape[dim]
                    self.creators[0].backward(self.grad.expand(dim,ds))

                if("expand" in self.creation_op):
                    dim = int(self.creation_op.split("_")[1])
                    self.creators[0].backward(self.grad.sum(dim))

    def __add__(self, other):
        if(self.autograd and other.autograd):
            return Tensor(self.data + other.data,
                          autograd=True,
                          creators=[self,other],
                          creation_op="add")
        return Tensor(self.data + other.data)

    def __neg__(self):
        if(self.autograd):
            return Tensor(self.data * -1,
                          autograd=True,
                          creators=[self],
                          creation_op="neg")
        return Tensor(self.data * -1)

    def __sub__(self, other):
        if(self.autograd and other.autograd):
            return Tensor(self.data - other.data,
                          autograd=True,
                          creators=[self,other],
                          creation_op="sub")
        return Tensor(self.data - other.data)

    def __mul__(self, other):
        if(self.autograd and other.autograd):
            return Tensor(self.data * other.data,
                          autograd=True,
                          creators=[self,other],
                          creation_op="mul")
        return Tensor(self.data * other.data)

    def sum(self, dim):
        if(self.autograd):
            return Tensor(self.data.sum(dim),
                          autograd=True,
                          creators=[self],
                          creation_op="sum_"+str(dim))
        return Tensor(self.data.sum(dim))

    def expand(self, dim,copies):

        trans_cmd = list(range(0,len(self.data.shape)))
        trans_cmd.insert(dim,len(self.data.shape))
        new_shape = list(self.data.shape) + [copies]
        new_data = self.data.repeat(copies).reshape(new_shape)
        new_data = new_data.transpose(trans_cmd)

        if(self.autograd):
            return Tensor(new_data,
                          autograd=True,
                          creators=[self],
                          creation_op="expand_"+str(dim))
        return Tensor(new_data)

    def transpose(self):
        if(self.autograd):
            return Tensor(self.data.transpose(),
                          autograd=True,
                          creators=[self],
                          creation_op="transpose")

        return Tensor(self.data.transpose())

    def mm(self, x):
        if(self.autograd):
            return Tensor(self.data.dot(x.data),
                          autograd=True,
                          creators=[self,x],
                          creation_op="mm")
        return Tensor(self.data.dot(x.data))

    def __repr__(self):
        return str(self.data.__repr__())

    def __str__(self):
        return str(self.data.__str__())

### **9.1 `sum` and `expand`**

The derivatives of all these functions have been discussed before, but `sum` and `expand` may look unfamiliar because of their names. **`sum(dim)`** adds up the entries across one dimension of the tensor. Take a 2 x 3 matrix `x`:

In [10]:
x = Tensor(np.array([[1,2,3],
                     [4,5,6]]))

`x.sum(0)` sums across the first dimension and gives the length-3 vector `[5, 7, 9]`, whereas `x.sum(1)` sums across the second dimension and gives the length-2 vector `[6, 15]`.

In [11]:
x.sum(0)

array([5, 7, 9])

In [12]:
x.sum(1)

array([ 6, 15])

**`expand(dim, copies)`** is the function used to backpropagate through a `sum`: it copies data along a dimension. Whereas `sum` *removes* a dimension, `expand` *adds* one. Copying the matrix `x` along the first dimension four times gives four stacked copies, so the 2 x 3 matrix becomes a 4 x 2 x 3 tensor (a list of four 2 x 3 tensors).

In [13]:
x.expand(dim=0, copies=4)

array([[[1, 2, 3],
        [4, 5, 6]],

       [[1, 2, 3],
        [4, 5, 6]],

       [[1, 2, 3],
        [4, 5, 6]],

       [[1, 2, 3],
        [4, 5, 6]]])

Expanding along the last dimension copies *inside* the tensor instead: every entry of the original tensor turns into a list of four equal entries, and the result has shape 2 x 3 x 4.

In [14]:
x.expand(dim=2, copies=4)

array([[[1, 1, 1, 1],
        [2, 2, 2, 2],
        [3, 3, 3, 3]],

       [[4, 4, 4, 4],
        [5, 5, 5, 5],
        [6, 6, 6, 6]]])

The rule that results is: when `.sum(dim=1)` is applied to a tensor that has four entries in that dimension, the gradient must be backpropagated with `.expand(dim=1, copies=4)`. Conversely, the gradient of an `expand` is obtained with `sum` over the same dimension.

### **9.2 The Backpropagation Logic**

The new `backward` branches are the following (they are part of cell v5 above):

| Operation | Gradient sent to the creators |
|---|---|
| `sub` | the gradient itself to the first creator, its negation to the second |
| `mul` | the gradient multiplied by the *other* creator (elementwise) |
| `mm` | to the activation (usually `creators[0]`): `grad.mm(weights.transpose())`; to the weight matrix (usually `creators[1]`): `grad.transpose().mm(act).transpose()` |
| `transpose` | the transposed gradient |
| `sum_<dim>` | the gradient expanded along `dim` by the size of that dimension in the creator |
| `expand_<dim>` | the gradient summed along `dim` |

The branches `neg`, `mm`, `transpose`, `sum`, and `expand` call `.backward()` without passing `self` as `grad_origin`, whereas `add`, `sub`, and `mul` pass it; the book does not comment on this difference.

If any of this feels uncertain, the best remedy is to revisit the way backpropagation was done in Chapter 6, which has figures showing each step. The gradients start at the end of the network and the error signal is moved backward by calling functions that correspond to the ones used to move activations forward. If the last forward operation was a matrix multiplication (and it was), one backpropagates with a matrix multiplication by the *transposed* matrix. In the Chapter 6 code this is the line `layer_1_delta=layer_2_delta.dot(weights_1_2.T)` of the first figure below, and in the new code it is the line `new = self.grad.mm(weights.transpose())` of the `mm` branch. The operations are exactly the same, performed in reverse order of forward propagation, but the code is better organized.

> Chapter 6 revisited, step 3: backpropagating from `layer_2` to `layer_1` with `layer_1_delta=layer_2_delta.dot(weights_1_2.T)`; this corresponds to the `mm` branch of `backward`.
>
> <img src="pictures/chapter 13/picture 13.3.png" alt="Figure 13.3" width="600"/>

> Chapter 6 revisited, step 4: generating the `weight_deltas` and updating the weights.
>
> <img src="pictures/chapter 13/picture 13.4.png" alt="Figure 13.4" width="600"/>


---
## **10. Using Autograd to Train a Neural Network**

**You no longer have to write backpropagation logic.**

The engineering effort pays off now: training a network no longer requires any backpropagation code. As a toy example, the book first shows a network whose backpropagation is written by hand. It has two inputs, a hidden layer of three units, and one output (no nonlinearity), and it is trained on four examples with the targets 0, 1, 0, and 1. The loss is the squared error summed over the examples (the book labels it mean squared error loss).

### **10.1 The Hand-Coded Version**

The forward pass has to keep `layer_1`, `layer_2`, and `diff` in variables because the backward pass needs them. Each gradient is then propagated to its weight matrix by hand, and the weights are updated.

*Note on the printed output:* the book prints the ten loss values `0.452...` (first) down to `0.0523...` (last) below this listing. Run exactly as printed (with `np.random.seed(0)`), the cell prints values starting at about 5.07 and ending at about 0.116, so the book's printed numbers could not be reproduced.

In [15]:
import numpy
np.random.seed(0)

data = np.array([[0,0],[0,1],[1,0],[1,1]])
target = np.array([[0],[1],[0],[1]])

weights_0_1 = np.random.rand(2,3)
weights_1_2 = np.random.rand(3,1)

for i in range(10):

    layer_1 = data.dot(weights_0_1)
    layer_2 = layer_1.dot(weights_1_2)

    diff = (layer_2 - target)
    sqdiff = (diff * diff)
    loss = sqdiff.sum(0)

    layer_1_grad = diff.dot(weights_1_2.transpose())
    weight_1_2_update = layer_1.transpose().dot(diff)
    weight_0_1_update = data.transpose().dot(layer_1_grad)

    weights_1_2 -= weight_1_2_update * 0.1
    weights_0_1 -= weight_0_1_update * 0.1
    print(loss[0])

5.066439994622396
0.4959907791902341
0.4180671892167177
0.35298133007809646
0.2972549636567376
0.24923260381633278
0.20785392075862477
0.17231260916265181
0.14193744536652994
0.11613979792168387


### **10.2 The Version with Autograd**

With the new autograd system, the same network becomes much simpler. The parameters are kept in a list `w`; the prediction is `data.mm(w[0]).mm(w[1])`; the loss is `((pred - target)*(pred - target)).sum(0)`; and the learning step is `loss.backward(Tensor(np.ones_like(loss.data)))` followed by a loop that updates every weight with `w_.data -= w_.grad.data * 0.1` and then resets its gradient with `w_.grad.data *= 0`. There is no need to keep temporary variables (the dynamic graph remembers them) and no backpropagation logic to implement (`.backward()` handles it). This is not only more convenient but also reduces the risk of silly mistakes in the backpropagation code.

The book prints the loss decreasing from `[0.58128304]` to `[0.06849361]` over the ten iterations. The two programs do not start from the same random weights, so their loss values are different.

In [16]:
import numpy
np.random.seed(0)

data = Tensor(np.array([[0,0],[0,1],[1,0],[1,1]]), autograd=True)
target = Tensor(np.array([[0],[1],[0],[1]]), autograd=True)

w = list()
w.append(Tensor(np.random.rand(2,3), autograd=True))
w.append(Tensor(np.random.rand(3,1), autograd=True))

for i in range(10):

    pred = data.mm(w[0]).mm(w[1])

    loss = ((pred - target)*(pred - target)).sum(0)

    loss.backward(Tensor(np.ones_like(loss.data)))

    for w_ in w:
        w_.data -= w_.grad.data * 0.1
        w_.grad.data *= 0

    print(loss)

[0.58128304]
[0.48988149]
[0.41375111]
[0.34489412]
[0.28210124]
[0.2254484]
[0.17538853]
[0.1324231]
[0.09682769]
[0.06849361]


One stylistic detail deserves attention: all parameters were put in a list, which is iterated over when the weights are updated. This foreshadows the next piece of functionality. With an autograd system, **stochastic gradient descent becomes trivial** to implement (it is just the final `for` loop), so the next step is to turn it into a class of its own.

---
## **11. Adding Automatic Optimization**

**A stochastic gradient descent optimizer is only a small piece of object-oriented programming.**

Creating an "SGD optimizer" sounds difficult, but it consists of copying the update loop of the previous example into a class:

- `SGD(parameters, alpha=0.1)` stores the list of parameters (tensors) and the learning rate `alpha`;
- `zero()` sets the gradient of every parameter to zero;
- `step(zero=True)` updates every parameter with `p.data -= p.grad.data * self.alpha` and, unless `zero=False` is passed, resets its gradient to zero afterwards.

In [17]:
class SGD(object):

    def __init__(self, parameters, alpha=0.1):
        self.parameters = parameters
        self.alpha = alpha

    def zero(self):
        for p in self.parameters:
            p.grad.data *= 0

    def step(self, zero=True):

        for p in self.parameters:

            p.data -= p.grad.data * self.alpha

            if(zero):
                p.grad.data *= 0

The network of the previous section is simplified as follows. The weight update loop is replaced by `optim.step()`, and the book states that the results are exactly the same as before. (The book prints no `print` statement in this listing, so the cell shows no output.)

In [18]:
import numpy
np.random.seed(0)

data = Tensor(np.array([[0,0],[0,1],[1,0],[1,1]]), autograd=True)
target = Tensor(np.array([[0],[1],[0],[1]]), autograd=True)

w = list()
w.append(Tensor(np.random.rand(2,3), autograd=True))
w.append(Tensor(np.random.rand(3,1), autograd=True))

optim = SGD(parameters=w, alpha=0.1)

for i in range(10):

    pred = data.mm(w[0]).mm(w[1])

    loss = ((pred - target)*(pred - target)).sum(0)

    loss.backward(Tensor(np.ones_like(loss.data)))
    optim.step()

---
## **12. Adding Support for Layer Types**

**Layers package common forward-propagation steps behind a simple `.forward()` interface.**

The most complicated pieces of the framework are done. What remains is mostly about adding new functions to the tensor and creating convenient higher-order classes and functions. The most common abstraction in nearly all frameworks (familiar from Keras and PyTorch) is the **layer**: a collection of commonly used forward propagation techniques packaged into a simple API with a `.forward()` method.

### **12.1 A Linear Layer**

- `Layer` is an abstract class with a list `parameters` and a single getter, `get_parameters()`. More complicated layer types (for example layers containing other layers) only have to override `get_parameters()` to control which tensors are later handed to the optimizer (such as the `SGD` class).
- `Linear(n_inputs, n_outputs)` is a true linear layer: the weights are initialized with `np.random.randn(n_inputs, n_outputs)*np.sqrt(2.0/(n_inputs))`, and a **bias** vector of zeros is added. Both are tensors with `autograd=True` and are appended to `self.parameters`. In `forward`, the input is multiplied by the weight matrix and the bias is added after being expanded with `expand(0,len(input.data))`, i.e., once for every row of the input.

Nothing here is particularly new: the weights are simply organized into a class, so that weights and bias are initialized with the correct sizes and the correct forward logic is always used.

In [19]:
class Layer(object):

    def __init__(self):
        self.parameters = list()

    def get_parameters(self):
        return self.parameters


class Linear(Layer):

    def __init__(self, n_inputs, n_outputs):
        super().__init__()
        W = np.random.randn(n_inputs, n_outputs)*np.sqrt(2.0/(n_inputs))
        self.weight = Tensor(W, autograd=True)
        self.bias = Tensor(np.zeros(n_outputs), autograd=True)

        self.parameters.append(self.weight)
        self.parameters.append(self.bias)

    def forward(self, input):
        return input.mm(self.weight)+self.bias.expand(0,len(input.data))

### **12.2 Layers That Contain Layers**

The most popular container is the **sequential layer**, which forward propagates a list of layers, feeding the output of each layer into the next. `Sequential` stores the list in `self.layers`, offers `add(layer)` to append a layer, loops over the layers in `forward`, and overrides `get_parameters()` so that it collects the parameters of all the layers it contains. The listing below defines the class and then trains a model `Sequential([Linear(2,3), Linear(3,1)])` on the same data with `SGD` and `alpha=0.05`, printing the loss. (The book sets no random seed in this listing and prints no output for it, so the values change from run to run.)

In [20]:
class Sequential(Layer):

    def __init__(self, layers=list()):
        super().__init__()

        self.layers = layers

    def add(self, layer):
        self.layers.append(layer)

    def forward(self, input):
        for layer in self.layers:
            input = layer.forward(input)
        return input

    def get_parameters(self):
        params = list()
        for l in self.layers:
            params += l.get_parameters()
        return params

data = Tensor(np.array([[0,0],[0,1],[1,0],[1,1]]), autograd=True)
target = Tensor(np.array([[0],[1],[0],[1]]), autograd=True)

model = Sequential([Linear(2,3), Linear(3,1)])

optim = SGD(parameters=model.get_parameters(), alpha=0.05)

for i in range(10):

    pred = model.forward(data)

    loss = ((pred - target)*(pred - target)).sum(0)

    loss.backward(Tensor(np.ones_like(loss.data)))
    optim.step()
    print(loss)

[2.6865042]
[11.271212]
[38.09060898]
[9.34796148]
[2.28355956]
[0.93643926]
[0.61224936]
[0.43932735]
[0.32808561]
[0.24902135]


### **12.3 Loss-Function Layers**

Some layers have **no weights** and are just functions of their input. The best-known example is the loss-function layer, such as **mean squared error**: `MSELoss.forward(pred, target)` returns `((pred - target)*(pred - target)).sum(0)`. The training listing now builds the model with `Sequential`, uses `criterion = MSELoss()`, and calls `criterion.forward(pred, target)`. The book prints the losses `[2.33428272]`, `[0.06743796]`, ..., `[0.01153118]`, `[0.00889602]`.

Once more, nothing is particularly new: under the hood the last several listings perform the same computation. The difference is that autograd does the backpropagation and that the forward steps are packaged into classes that make sure they are executed in the right order.

In [21]:
class MSELoss(Layer):

    def __init__(self):
        super().__init__()

    def forward(self, pred, target):
        return ((pred - target)*(pred - target)).sum(0)

import numpy
np.random.seed(0)

data = Tensor(np.array([[0,0],[0,1],[1,0],[1,1]]), autograd=True)
target = Tensor(np.array([[0],[1],[0],[1]]), autograd=True)

model = Sequential([Linear(2,3), Linear(3,1)])
criterion = MSELoss()

optim = SGD(parameters=model.get_parameters(), alpha=0.05)

for i in range(10):

    pred = model.forward(data)

    loss = criterion.forward(pred, target)

    loss.backward(Tensor(np.ones_like(loss.data)))
    optim.step()
    print(loss)

[2.33428272]
[0.06743796]
[0.0521849]
[0.04079507]
[0.03184365]
[0.02479336]
[0.01925443]
[0.01491699]
[0.01153118]
[0.00889602]


---
## **13. How to Learn a Framework**

**Oversimplified, a framework is autograd plus a list of prebuilt layers and optimizers.**

New layer types could be written quickly on top of the autograd system, which is the main feature of modern frameworks: it removes the need to hand-write every math operation for forward and backward propagation. Frameworks therefore increase the speed from idea to experiment and reduce the number of bugs.

Viewing a framework as an autograd system plus a long list of layers and optimizers makes frameworks easier to learn. The author expects the reader to be able to move from this chapter to almost any framework fairly quickly, and notes that the one whose API is most similar to the framework built here is **PyTorch**. For reference, the book recommends looking through the lists of layers and optimizers in the documentation of several big frameworks:

- https://pytorch.org/docs/stable/nn.html
- https://keras.io/layers/about-keras-layers
- https://www.tensorflow.org/api_docs/python/tf/layers

The general workflow for learning a new framework is: find the simplest possible code example, tweak it to get to know the API of the autograd system, and then modify the example piece by piece until it becomes the experiment one cares about.

### **13.1 A Convenience in `backward`**

One more addition is made to `Tensor.backward()`: if no gradient is passed, a gradient of ones is created, so the user no longer has to pass a gradient of ones the first time `.backward()` is called. It is not strictly necessary, but it is handy:

```python
def backward(self,grad=None, grad_origin=None):
    if(self.autograd):

        if(grad is None):
            grad = Tensor(np.ones_like(self.data))
```

This fragment is not a separate cell: it is included in the next version of the class (v6, in the next section), which also contains the new nonlinearities. Nothing between the two places uses it.

---
## **14. Nonlinearity Layers**

**Nonlinear functions are added to `Tensor` first, and layers that use them follow.**

The next chapter needs `.sigmoid()` and `.tanh()`, so they are added to the `Tensor` class. Their derivatives were learned long ago, so this is routine:

- `sigmoid` computes `1 / (1 + np.exp(-self.data))` and, in `backward`, sends `self.grad * (self * (ones - self))`;
- `tanh` computes `np.tanh(self.data)` and, in `backward`, sends `self.grad * (ones - (self * self))`,

where `ones` is a tensor of ones with the shape of the gradient. The reader is invited to create further nonlinearities, for example HardTanh or ReLU.

Cell v6 contains the complete class with these two functions (and the default gradient of the previous section). It replaces the former class.

In [22]:
import numpy as np
class Tensor (object):

    def __init__(self,data,
                 autograd=False,
                 creators=None,
                 creation_op=None,
                 id=None):

        self.data = np.array(data)
        self.creators = creators
        self.creation_op = creation_op
        self.grad = None
        self.autograd = autograd
        self.children = {}
        if(id is None):
            id = np.random.randint(0,100000)
        self.id = id

        if(creators is not None):
            for c in creators:
                if(self.id not in c.children):
                    c.children[self.id] = 1
                else:
                    c.children[self.id] += 1

    def all_children_grads_accounted_for(self):
        for id,cnt in self.children.items():
            if(cnt != 0):
                return False
        return True

    def backward(self,grad=None, grad_origin=None):
        if(self.autograd):

            if(grad is None):
                grad = Tensor(np.ones_like(self.data))

            if(grad_origin is not None):
                if(self.children[grad_origin.id] == 0):
                    raise Exception("cannot backprop more than once")
                else:
                    self.children[grad_origin.id] -= 1

            if(self.grad is None):
                self.grad = grad
            else:
                self.grad += grad

            if(self.creators is not None and
               (self.all_children_grads_accounted_for() or
                grad_origin is None)):

                if(self.creation_op == "add"):
                    self.creators[0].backward(self.grad, self)
                    self.creators[1].backward(self.grad, self)

                if(self.creation_op == "neg"):
                    self.creators[0].backward(self.grad.__neg__())

                if(self.creation_op == "sub"):
                    new = Tensor(self.grad.data)
                    self.creators[0].backward(new, self)
                    new = Tensor(self.grad.__neg__().data)
                    self.creators[1].backward(new, self)

                if(self.creation_op == "mul"):
                    new = self.grad * self.creators[1]
                    self.creators[0].backward(new , self)
                    new = self.grad * self.creators[0]
                    self.creators[1].backward(new, self)

                if(self.creation_op == "mm"):
                    act = self.creators[0]
                    weights = self.creators[1]
                    new = self.grad.mm(weights.transpose())
                    act.backward(new)
                    new = self.grad.transpose().mm(act).transpose()
                    weights.backward(new)

                if(self.creation_op == "transpose"):
                    self.creators[0].backward(self.grad.transpose())

                if("sum" in self.creation_op):
                    dim = int(self.creation_op.split("_")[1])
                    ds = self.creators[0].data.shape[dim]
                    self.creators[0].backward(self.grad.expand(dim,ds))

                if("expand" in self.creation_op):
                    dim = int(self.creation_op.split("_")[1])
                    self.creators[0].backward(self.grad.sum(dim))

                if(self.creation_op == "sigmoid"):
                    ones = Tensor(np.ones_like(self.grad.data))
                    self.creators[0].backward(self.grad * (self * (ones - self)))

                if(self.creation_op == "tanh"):
                    ones = Tensor(np.ones_like(self.grad.data))
                    self.creators[0].backward(self.grad * (ones - (self * self)))

    def __add__(self, other):
        if(self.autograd and other.autograd):
            return Tensor(self.data + other.data,
                          autograd=True,
                          creators=[self,other],
                          creation_op="add")
        return Tensor(self.data + other.data)

    def __neg__(self):
        if(self.autograd):
            return Tensor(self.data * -1,
                          autograd=True,
                          creators=[self],
                          creation_op="neg")
        return Tensor(self.data * -1)

    def __sub__(self, other):
        if(self.autograd and other.autograd):
            return Tensor(self.data - other.data,
                          autograd=True,
                          creators=[self,other],
                          creation_op="sub")
        return Tensor(self.data - other.data)

    def __mul__(self, other):
        if(self.autograd and other.autograd):
            return Tensor(self.data * other.data,
                          autograd=True,
                          creators=[self,other],
                          creation_op="mul")
        return Tensor(self.data * other.data)

    def sum(self, dim):
        if(self.autograd):
            return Tensor(self.data.sum(dim),
                          autograd=True,
                          creators=[self],
                          creation_op="sum_"+str(dim))
        return Tensor(self.data.sum(dim))

    def expand(self, dim,copies):

        trans_cmd = list(range(0,len(self.data.shape)))
        trans_cmd.insert(dim,len(self.data.shape))
        new_shape = list(self.data.shape) + [copies]
        new_data = self.data.repeat(copies).reshape(new_shape)
        new_data = new_data.transpose(trans_cmd)

        if(self.autograd):
            return Tensor(new_data,
                          autograd=True,
                          creators=[self],
                          creation_op="expand_"+str(dim))
        return Tensor(new_data)

    def transpose(self):
        if(self.autograd):
            return Tensor(self.data.transpose(),
                          autograd=True,
                          creators=[self],
                          creation_op="transpose")

        return Tensor(self.data.transpose())

    def mm(self, x):
        if(self.autograd):
            return Tensor(self.data.dot(x.data),
                          autograd=True,
                          creators=[self,x],
                          creation_op="mm")
        return Tensor(self.data.dot(x.data))

    def sigmoid(self):
        if(self.autograd):
            return Tensor(1 / (1 + np.exp(-self.data)),
                          autograd=True,
                          creators=[self],
                          creation_op="sigmoid")
        return Tensor(1 / (1 + np.exp(-self.data)))

    def tanh(self):
        if(self.autograd):
            return Tensor(np.tanh(self.data),
                          autograd=True,
                          creators=[self],
                          creation_op="tanh")
        return Tensor(np.tanh(self.data))

    def __repr__(self):
        return str(self.data.__repr__())

    def __str__(self):
        return str(self.data.__str__())

The two matching layer classes simply call the tensor functions (the book prints them side by side):

In [23]:
class Tanh(Layer):
    def __init__(self):
        super().__init__()

    def forward(self, input):
        return input.tanh()


class Sigmoid(Layer):
    def __init__(self):
        super().__init__()

    def forward(self, input):
        return input.sigmoid()

Now the new nonlinearities are tried out. The model becomes `Sequential([Linear(2,3), Tanh(), Linear(3,1), Sigmoid()])`, trained with `alpha=1`. The book prints the loss falling from `[1.06372865]` to `[0.04700013]` over ten iterations. The layers are dropped into the list given to `Sequential()`, and the network knows exactly how to use them.

In [24]:
import numpy
np.random.seed(0)

data = Tensor(np.array([[0,0],[0,1],[1,0],[1,1]]), autograd=True)
target = Tensor(np.array([[0],[1],[0],[1]]), autograd=True)

model = Sequential([Linear(2,3), Tanh(), Linear(3,1), Sigmoid()])
criterion = MSELoss()

optim = SGD(parameters=model.get_parameters(), alpha=1)

for i in range(10):

    pred = model.forward(data)

    loss = criterion.forward(pred, target)

    loss.backward(Tensor(np.ones_like(loss.data)))
    optim.step()
    print(loss)

[1.06372865]
[0.75148144]
[0.57384259]
[0.39574294]
[0.2482279]
[0.15515294]
[0.10423398]
[0.07571169]
[0.05837623]
[0.04700013]


The previous chapter introduced recurrent neural networks, and a model was trained to predict the next word given the previous several words. The author now asks the reader to translate that code into the new framework. Three new layer types are required: an **embedding layer** that learns word embeddings, an **RNN layer** that models sequences of inputs, and a layer that predicts a probability distribution over labels (a softmax, which will be handled through the cross-entropy loss).

---
## **15. The Embedding Layer**

**An embedding layer translates indices into activations.**

### **15.1 The Initial Specification**

Chapter 11 introduced word embeddings: vectors that are mapped to words and can be forward propagated into a network. With a vocabulary of 200 words there are 200 embeddings. The first step in creating the layer is to initialize a matrix with one row (vector) per word in the vocabulary and one column per embedding dimension: `weight = (np.random.rand(vocab_size, dim) - 0.5) / dim`. The book remarks that this initialization style is a convention from word2vec.

*Note on a repair:* the book prints `weight = np.random.rand(vocab_size, dim) - 0.5) / dim`, which has an unbalanced parenthesis; this notebook uses `weight = (np.random.rand(vocab_size, dim) - 0.5) / dim`. This first version is only the initial specification (it has no `forward` yet) and is replaced by the version of section 15.3.

In [25]:
class Embedding(Layer):

    def __init__(self, vocab_size, dim):
        super().__init__()

        self.vocab_size = vocab_size
        self.dim = dim

        weight = (np.random.rand(vocab_size, dim) - 0.5) / dim

Forward propagation starts with the question of how the inputs are encoded. Words themselves cannot be passed in, because they do not tell which row of `self.weight` to use; as recalled from Chapter 11, **indices** are forward propagated instead. NumPy supports this directly. Indexing the identity matrix `np.eye(5)` with a matrix of integers returns the same matrix with each integer replaced by the corresponding row, so a 2D matrix of indices becomes a 3D matrix of embeddings (rows).

In [26]:
identity = np.eye(5)
print(identity)

[[1. 0. 0. 0. 0.]
 [0. 1. 0. 0. 0.]
 [0. 0. 1. 0. 0.]
 [0. 0. 0. 1. 0.]
 [0. 0. 0. 0. 1.]]


In [27]:
print(identity[np.array([[1,2,3,4],
                         [2,3,4,0]])])

[[[0. 1. 0. 0. 0.]
  [0. 0. 1. 0. 0.]
  [0. 0. 0. 1. 0.]
  [0. 0. 0. 0. 1.]]

 [[0. 0. 1. 0. 0.]
  [0. 0. 0. 1. 0.]
  [0. 0. 0. 0. 1.]
  [1. 0. 0. 0. 0.]]]


### **15.2 Adding Indexing to Autograd**

Before the embedding layer can be completed, autograd has to support indexing. During backpropagation, the gradients must be placed in the same rows that were selected in forward propagation, which means the indices used in the forward pass have to be remembered. The new method `index_select(indices)` selects `self.data[indices.data]` and, if `autograd` is on, stores the indices in `new.index_select_indices`.

The matching `backward` branch works as follows:

1. Create a new gradient of the correct size, namely the size of the original matrix that was indexed (`np.zeros_like(self.creators[0].data)`).
2. Flatten the indices so that they can be iterated over.
3. Reshape the incoming gradient `grad_` into a simple list of rows (the order of the indices and of these row vectors correspond).
4. Loop over the indices and **add** each gradient row into the matching row of the new gradient, which is finally backpropagated into `self.creators[0]`.

Cell v7 is the complete class with `index_select` and its `backward` branch. It replaces the previous class. In `backward` the branch uses the argument `grad` (not `self.grad`).

In [28]:
import numpy as np
class Tensor (object):

    def __init__(self,data,
                 autograd=False,
                 creators=None,
                 creation_op=None,
                 id=None):

        self.data = np.array(data)
        self.creators = creators
        self.creation_op = creation_op
        self.grad = None
        self.autograd = autograd
        self.children = {}
        if(id is None):
            id = np.random.randint(0,100000)
        self.id = id

        if(creators is not None):
            for c in creators:
                if(self.id not in c.children):
                    c.children[self.id] = 1
                else:
                    c.children[self.id] += 1

    def all_children_grads_accounted_for(self):
        for id,cnt in self.children.items():
            if(cnt != 0):
                return False
        return True

    def backward(self,grad=None, grad_origin=None):
        if(self.autograd):

            if(grad is None):
                grad = Tensor(np.ones_like(self.data))

            if(grad_origin is not None):
                if(self.children[grad_origin.id] == 0):
                    raise Exception("cannot backprop more than once")
                else:
                    self.children[grad_origin.id] -= 1

            if(self.grad is None):
                self.grad = grad
            else:
                self.grad += grad

            if(self.creators is not None and
               (self.all_children_grads_accounted_for() or
                grad_origin is None)):

                if(self.creation_op == "add"):
                    self.creators[0].backward(self.grad, self)
                    self.creators[1].backward(self.grad, self)

                if(self.creation_op == "neg"):
                    self.creators[0].backward(self.grad.__neg__())

                if(self.creation_op == "sub"):
                    new = Tensor(self.grad.data)
                    self.creators[0].backward(new, self)
                    new = Tensor(self.grad.__neg__().data)
                    self.creators[1].backward(new, self)

                if(self.creation_op == "mul"):
                    new = self.grad * self.creators[1]
                    self.creators[0].backward(new , self)
                    new = self.grad * self.creators[0]
                    self.creators[1].backward(new, self)

                if(self.creation_op == "mm"):
                    act = self.creators[0]
                    weights = self.creators[1]
                    new = self.grad.mm(weights.transpose())
                    act.backward(new)
                    new = self.grad.transpose().mm(act).transpose()
                    weights.backward(new)

                if(self.creation_op == "transpose"):
                    self.creators[0].backward(self.grad.transpose())

                if("sum" in self.creation_op):
                    dim = int(self.creation_op.split("_")[1])
                    ds = self.creators[0].data.shape[dim]
                    self.creators[0].backward(self.grad.expand(dim,ds))

                if("expand" in self.creation_op):
                    dim = int(self.creation_op.split("_")[1])
                    self.creators[0].backward(self.grad.sum(dim))

                if(self.creation_op == "sigmoid"):
                    ones = Tensor(np.ones_like(self.grad.data))
                    self.creators[0].backward(self.grad * (self * (ones - self)))

                if(self.creation_op == "tanh"):
                    ones = Tensor(np.ones_like(self.grad.data))
                    self.creators[0].backward(self.grad * (ones - (self * self)))

                if(self.creation_op == "index_select"):
                    new_grad = np.zeros_like(self.creators[0].data)
                    indices_ = self.index_select_indices.data.flatten()
                    grad_ = grad.data.reshape(len(indices_), -1)
                    for i in range(len(indices_)):
                        new_grad[indices_[i]] += grad_[i]
                    self.creators[0].backward(Tensor(new_grad))

    def __add__(self, other):
        if(self.autograd and other.autograd):
            return Tensor(self.data + other.data,
                          autograd=True,
                          creators=[self,other],
                          creation_op="add")
        return Tensor(self.data + other.data)

    def __neg__(self):
        if(self.autograd):
            return Tensor(self.data * -1,
                          autograd=True,
                          creators=[self],
                          creation_op="neg")
        return Tensor(self.data * -1)

    def __sub__(self, other):
        if(self.autograd and other.autograd):
            return Tensor(self.data - other.data,
                          autograd=True,
                          creators=[self,other],
                          creation_op="sub")
        return Tensor(self.data - other.data)

    def __mul__(self, other):
        if(self.autograd and other.autograd):
            return Tensor(self.data * other.data,
                          autograd=True,
                          creators=[self,other],
                          creation_op="mul")
        return Tensor(self.data * other.data)

    def sum(self, dim):
        if(self.autograd):
            return Tensor(self.data.sum(dim),
                          autograd=True,
                          creators=[self],
                          creation_op="sum_"+str(dim))
        return Tensor(self.data.sum(dim))

    def expand(self, dim,copies):

        trans_cmd = list(range(0,len(self.data.shape)))
        trans_cmd.insert(dim,len(self.data.shape))
        new_shape = list(self.data.shape) + [copies]
        new_data = self.data.repeat(copies).reshape(new_shape)
        new_data = new_data.transpose(trans_cmd)

        if(self.autograd):
            return Tensor(new_data,
                          autograd=True,
                          creators=[self],
                          creation_op="expand_"+str(dim))
        return Tensor(new_data)

    def transpose(self):
        if(self.autograd):
            return Tensor(self.data.transpose(),
                          autograd=True,
                          creators=[self],
                          creation_op="transpose")

        return Tensor(self.data.transpose())

    def mm(self, x):
        if(self.autograd):
            return Tensor(self.data.dot(x.data),
                          autograd=True,
                          creators=[self,x],
                          creation_op="mm")
        return Tensor(self.data.dot(x.data))

    def sigmoid(self):
        if(self.autograd):
            return Tensor(1 / (1 + np.exp(-self.data)),
                          autograd=True,
                          creators=[self],
                          creation_op="sigmoid")
        return Tensor(1 / (1 + np.exp(-self.data)))

    def tanh(self):
        if(self.autograd):
            return Tensor(np.tanh(self.data),
                          autograd=True,
                          creators=[self],
                          creation_op="tanh")
        return Tensor(np.tanh(self.data))

    def index_select(self, indices):

        if(self.autograd):
            new = Tensor(self.data[indices.data],
                         autograd=True,
                         creators=[self],
                         creation_op="index_select")
            new.index_select_indices = indices
            return new
        return Tensor(self.data[indices.data])

    def __repr__(self):
        return str(self.data.__repr__())

    def __str__(self):
        return str(self.data.__str__())

The test below selects rows 1, 2, 3 and 2, 3, 4 of the 5 x 5 identity matrix and backpropagates without passing a gradient (this uses the default gradient of ones from v6). Each used index receives a row of ones for every time it is used, so the rows with indices 2 and 3, which are used twice, get a gradient of 2, the rows 1 and 4 get 1, and row 0 (never selected) stays 0.

In [29]:
x = Tensor(np.eye(5), autograd=True)
x.index_select(Tensor([[1,2,3],
                       [2,3,4]])).backward()
print(x.grad)

[[0. 0. 0. 0. 0.]
 [1. 1. 1. 1. 1.]
 [2. 2. 2. 2. 2.]
 [2. 2. 2. 2. 2.]
 [1. 1. 1. 1. 1.]]


### **15.3 The Embedding Layer (Revisited)**

With `index_select` available, forward propagation is a single call, `self.weight.index_select(input)`, and autograd handles the rest. The layer stores the weight as a tensor with `autograd=True` and appends it to `self.parameters`.

*Note on a repair:* in addition to the unbalanced parenthesis of the previous version, the book prints `self.weight = Tensor((weight, autograd=True)`, which cannot be parsed; this notebook uses `self.weight = Tensor(weight, autograd=True)`.

The example below trains `Sequential([embed, Tanh(), Linear(3,1), Sigmoid()])` with `embed = Embedding(5,3)`, the input indices `[1,2,1,2]`, the targets `[[0],[1],[0],[1]]`, and `alpha=0.5`. The network learns to associate the input indices 1 and 2 with the predictions 0 and 1. In theory these indices could stand for words (or any other input objects), and in the final example of the chapter they will; here the aim is merely to show the embedding layer working. The book prints a loss that falls from about `0.989` to about `0.074`; this listing sets no random seed, so the numbers in your run will differ.

In [30]:
class Embedding(Layer):

    def __init__(self, vocab_size, dim):
        super().__init__()

        self.vocab_size = vocab_size
        self.dim = dim

        weight = (np.random.rand(vocab_size, dim) - 0.5) / dim
        self.weight = Tensor(weight, autograd=True)

        self.parameters.append(self.weight)

    def forward(self, input):
        return self.weight.index_select(input)


data = Tensor(np.array([1,2,1,2]), autograd=True)
target = Tensor(np.array([[0],[1],[0],[1]]), autograd=True)

embed = Embedding(5,3)
model = Sequential([embed, Tanh(), Linear(3,1), Sigmoid()])
criterion = MSELoss()

optim = SGD(parameters=model.get_parameters(), alpha=0.5)

for i in range(10):

    pred = model.forward(data)

    loss = criterion.forward(pred, target)

    loss.backward(Tensor(np.ones_like(loss.data)))
    optim.step()
    print(loss)

[0.89443309]
[0.5496876]
[0.35367458]
[0.23886296]
[0.17091404]
[0.12894201]
[0.10155089]
[0.08272372]
[0.0691961]
[0.05911161]


---
## **16. The Cross-Entropy Layer**

**Cross entropy is added to autograd as a tensor function and as a loss layer.**

By now the creation of new layer types should feel comfortable, and cross entropy is a standard one that has been used many times in this book. The code is therefore given for reference, with the suggestion to try writing it before copying it.

The method `cross_entropy(target_indices)` does the following:

- it computes the **softmax** of the data (`temp = np.exp(self.data)`, divided by its sum over the last axis);
- it flattens the target indices `t` and reshapes the softmax output into a matrix `p` with one row per target;
- it builds the one-hot target distribution `target_dist = np.eye(p.shape[1])[t]`;
- it computes the loss `-(np.log(p) * (target_dist)).sum(1).mean()`.

If `autograd` is on, the resulting tensor keeps `softmax_output` and `target_dist`, and its `backward` branch sends the gradient `dx = self.softmax_output - self.target_dist` to the creator. The class `CrossEntropyLoss` simply calls `input.cross_entropy(target)`.

Cell v8 is the complete class with `cross_entropy` and its `backward` branch; it replaces the previous class.

In [31]:
import numpy as np
class Tensor (object):

    def __init__(self,data,
                 autograd=False,
                 creators=None,
                 creation_op=None,
                 id=None):

        self.data = np.array(data)
        self.creators = creators
        self.creation_op = creation_op
        self.grad = None
        self.autograd = autograd
        self.children = {}
        if(id is None):
            id = np.random.randint(0,100000)
        self.id = id

        if(creators is not None):
            for c in creators:
                if(self.id not in c.children):
                    c.children[self.id] = 1
                else:
                    c.children[self.id] += 1

    def all_children_grads_accounted_for(self):
        for id,cnt in self.children.items():
            if(cnt != 0):
                return False
        return True

    def backward(self,grad=None, grad_origin=None):
        if(self.autograd):

            if(grad is None):
                grad = Tensor(np.ones_like(self.data))

            if(grad_origin is not None):
                if(self.children[grad_origin.id] == 0):
                    raise Exception("cannot backprop more than once")
                else:
                    self.children[grad_origin.id] -= 1

            if(self.grad is None):
                self.grad = grad
            else:
                self.grad += grad

            if(self.creators is not None and
               (self.all_children_grads_accounted_for() or
                grad_origin is None)):

                if(self.creation_op == "add"):
                    self.creators[0].backward(self.grad, self)
                    self.creators[1].backward(self.grad, self)

                if(self.creation_op == "neg"):
                    self.creators[0].backward(self.grad.__neg__())

                if(self.creation_op == "sub"):
                    new = Tensor(self.grad.data)
                    self.creators[0].backward(new, self)
                    new = Tensor(self.grad.__neg__().data)
                    self.creators[1].backward(new, self)

                if(self.creation_op == "mul"):
                    new = self.grad * self.creators[1]
                    self.creators[0].backward(new , self)
                    new = self.grad * self.creators[0]
                    self.creators[1].backward(new, self)

                if(self.creation_op == "mm"):
                    act = self.creators[0]
                    weights = self.creators[1]
                    new = self.grad.mm(weights.transpose())
                    act.backward(new)
                    new = self.grad.transpose().mm(act).transpose()
                    weights.backward(new)

                if(self.creation_op == "transpose"):
                    self.creators[0].backward(self.grad.transpose())

                if("sum" in self.creation_op):
                    dim = int(self.creation_op.split("_")[1])
                    ds = self.creators[0].data.shape[dim]
                    self.creators[0].backward(self.grad.expand(dim,ds))

                if("expand" in self.creation_op):
                    dim = int(self.creation_op.split("_")[1])
                    self.creators[0].backward(self.grad.sum(dim))

                if(self.creation_op == "sigmoid"):
                    ones = Tensor(np.ones_like(self.grad.data))
                    self.creators[0].backward(self.grad * (self * (ones - self)))

                if(self.creation_op == "tanh"):
                    ones = Tensor(np.ones_like(self.grad.data))
                    self.creators[0].backward(self.grad * (ones - (self * self)))

                if(self.creation_op == "index_select"):
                    new_grad = np.zeros_like(self.creators[0].data)
                    indices_ = self.index_select_indices.data.flatten()
                    grad_ = grad.data.reshape(len(indices_), -1)
                    for i in range(len(indices_)):
                        new_grad[indices_[i]] += grad_[i]
                    self.creators[0].backward(Tensor(new_grad))

                if(self.creation_op == "cross_entropy"):
                    dx = self.softmax_output - self.target_dist
                    self.creators[0].backward(Tensor(dx))

    def __add__(self, other):
        if(self.autograd and other.autograd):
            return Tensor(self.data + other.data,
                          autograd=True,
                          creators=[self,other],
                          creation_op="add")
        return Tensor(self.data + other.data)

    def __neg__(self):
        if(self.autograd):
            return Tensor(self.data * -1,
                          autograd=True,
                          creators=[self],
                          creation_op="neg")
        return Tensor(self.data * -1)

    def __sub__(self, other):
        if(self.autograd and other.autograd):
            return Tensor(self.data - other.data,
                          autograd=True,
                          creators=[self,other],
                          creation_op="sub")
        return Tensor(self.data - other.data)

    def __mul__(self, other):
        if(self.autograd and other.autograd):
            return Tensor(self.data * other.data,
                          autograd=True,
                          creators=[self,other],
                          creation_op="mul")
        return Tensor(self.data * other.data)

    def sum(self, dim):
        if(self.autograd):
            return Tensor(self.data.sum(dim),
                          autograd=True,
                          creators=[self],
                          creation_op="sum_"+str(dim))
        return Tensor(self.data.sum(dim))

    def expand(self, dim,copies):

        trans_cmd = list(range(0,len(self.data.shape)))
        trans_cmd.insert(dim,len(self.data.shape))
        new_shape = list(self.data.shape) + [copies]
        new_data = self.data.repeat(copies).reshape(new_shape)
        new_data = new_data.transpose(trans_cmd)

        if(self.autograd):
            return Tensor(new_data,
                          autograd=True,
                          creators=[self],
                          creation_op="expand_"+str(dim))
        return Tensor(new_data)

    def transpose(self):
        if(self.autograd):
            return Tensor(self.data.transpose(),
                          autograd=True,
                          creators=[self],
                          creation_op="transpose")

        return Tensor(self.data.transpose())

    def mm(self, x):
        if(self.autograd):
            return Tensor(self.data.dot(x.data),
                          autograd=True,
                          creators=[self,x],
                          creation_op="mm")
        return Tensor(self.data.dot(x.data))

    def sigmoid(self):
        if(self.autograd):
            return Tensor(1 / (1 + np.exp(-self.data)),
                          autograd=True,
                          creators=[self],
                          creation_op="sigmoid")
        return Tensor(1 / (1 + np.exp(-self.data)))

    def tanh(self):
        if(self.autograd):
            return Tensor(np.tanh(self.data),
                          autograd=True,
                          creators=[self],
                          creation_op="tanh")
        return Tensor(np.tanh(self.data))

    def index_select(self, indices):

        if(self.autograd):
            new = Tensor(self.data[indices.data],
                         autograd=True,
                         creators=[self],
                         creation_op="index_select")
            new.index_select_indices = indices
            return new
        return Tensor(self.data[indices.data])

    def cross_entropy(self, target_indices):

        temp = np.exp(self.data)
        softmax_output = temp / np.sum(temp,
                                       axis=len(self.data.shape)-1,
                                       keepdims=True)

        t = target_indices.data.flatten()
        p = softmax_output.reshape(len(t),-1)
        target_dist = np.eye(p.shape[1])[t]
        loss = -(np.log(p) * (target_dist)).sum(1).mean()

        if(self.autograd):
            out = Tensor(loss,
                         autograd=True,
                         creators=[self],
                         creation_op="cross_entropy")
            out.softmax_output = softmax_output
            out.target_dist = target_dist
            return out

        return Tensor(loss)

    def __repr__(self):
        return str(self.data.__repr__())

    def __str__(self):
        return str(self.data.__str__())

In [32]:
class CrossEntropyLoss(object):

    def __init__(self):
        super().__init__()

    def forward(self, input, target):
        return input.cross_entropy(target)

The example uses `data = Tensor(np.array([1,2,1,2]), autograd=True)` as input indices, `target = Tensor(np.array([0,1,0,1]), autograd=True)` as target indices, the model `Sequential([Embedding(3,3), Tanh(), Linear(3,4)])`, the cross-entropy criterion, and `alpha=0.1`. The book prints a loss that falls from `1.3885032434928422` to `0.16527389263866668` in ten iterations (the last digits may differ slightly with other NumPy versions).

In [33]:
import numpy
np.random.seed(0)

# data indices
data = Tensor(np.array([1,2,1,2]), autograd=True)

# target indices
target = Tensor(np.array([0,1,0,1]), autograd=True)

model = Sequential([Embedding(3,3), Tanh(), Linear(3,4)])
criterion = CrossEntropyLoss()

optim = SGD(parameters=model.get_parameters(), alpha=0.1)

for i in range(10):

    pred = model.forward(data)

    loss = criterion.forward(pred, target)

    loss.backward(Tensor(np.ones_like(loss.data)))
    optim.step()
    print(loss)

1.3885032434928422
0.9558181509266036
0.6823083585795604
0.509525996749312
0.39574491472895856
0.31752527285348287
0.26172228619642157
0.22061283923954228
0.18946427334830074
0.16527389263866668


One feature of this loss differs from the others: **both the final softmax and the computation of the loss are inside the loss class**. This is an extremely common convention in deep networks, and nearly every framework works this way. To finish a network and train it with cross entropy, one leaves the softmax out of the forward propagation and calls a cross-entropy class that performs the softmax as part of the loss function. The two are combined for **performance**: computing the gradient of softmax and negative log likelihood together is much faster than propagating forward and backward through two separate modules, thanks to a shortcut in the gradient math (here, `softmax_output - target_dist`).

---
## **17. The Recurrent Neural Network Layer**

**By combining several layers, one can learn over time series.**

The last exercise creates one more layer, composed of several smaller layers, in order to learn the task completed at the end of the previous chapter. This is the **recurrent layer**. It is built from three linear layers, and its `.forward()` method takes both the previous hidden state and the input from the current training data.

### **17.1 The `RNNCell` Layer**

It is out of scope to reintroduce RNNs here, but the pieces should look familiar. An RNN has a **state vector** that is passed from time step to time step; here it is the variable `hidden`, which is both an input parameter and an output of `forward`. An RNN also has several weight matrices: one that maps input vectors to hidden vectors (`self.w_ih`, the input-to-hidden layer), one that maps hidden to hidden (`self.w_hh`, which updates each hidden vector based on the previous one), and, optionally, one that maps the hidden vector to an output used for predictions (`self.w_ho`, the hidden-to-output layer). `RNNCell` includes all three. The dimensions are configured with `n_inputs`, `n_hidden`, and `n_output`; in the example below the cell receives embeddings of size 16 and produces one score for every word of the vocabulary.

The parameter `activation` selects the nonlinearity that is applied to the hidden vectors at each time step: the book adds two possibilities, `Sigmoid` and `Tanh` (there are many more to choose from). The method `forward(input, hidden)` returns the pair `(output, new_hidden)`, and `init_hidden(batch_size=1)` creates a tensor of zeros as the starting state.

*Note on a typo in the book:* in the `'tanh'` branch the book prints `self.activation == Tanh()` (a comparison) where an assignment `=` is meant. The cell below keeps the book's text, because the example of the chapter uses the default `'sigmoid'`; creating an `RNNCell` with `activation='tanh'` would fail with this line until `==` is changed to `=`.

In [34]:
class RNNCell(Layer):

    def __init__(self, n_inputs,n_hidden,n_output,activation='sigmoid'):
        super().__init__()

        self.n_inputs = n_inputs
        self.n_hidden = n_hidden
        self.n_output = n_output

        if(activation == 'sigmoid'):
            self.activation = Sigmoid()
        elif(activation == 'tanh'):
            self.activation == Tanh()
        else:
            raise Exception("Non-linearity not found")

        self.w_ih = Linear(n_inputs, n_hidden)
        self.w_hh = Linear(n_hidden, n_hidden)
        self.w_ho = Linear(n_hidden, n_output)

        self.parameters += self.w_ih.get_parameters()
        self.parameters += self.w_hh.get_parameters()
        self.parameters += self.w_ho.get_parameters()

    def forward(self, input, hidden):
        from_prev_hidden = self.w_hh.forward(hidden)
        combined = self.w_ih.forward(input) + from_prev_hidden
        new_hidden = self.activation.forward(combined)
        output = self.w_ho.forward(new_hidden)
        return output, new_hidden

    def init_hidden(self, batch_size=1):
        return Tensor(np.zeros((batch_size,self.n_hidden)),autograd=True)

### **17.2 Preparing the Data**

The network is trained on the same task as in the previous chapter. The listing reads the first 1000 lines of a training file, lowercases them, strips the line break, splits them into words, and drops the first token (the line number). Each line is left-padded with `'-'` up to 6 tokens. It then collects the vocabulary, builds a `word2index` dictionary, converts every line into a list of indices, and stores the result in the NumPy array `data`. (The function `words2indices` and some imports are printed but not used afterwards.)

*Data requirement:* this cell opens the file `tasksv11/en/qa1_single-supporting-fact_train.txt`, which must exist relative to the working directory. It is the training file of the single-supporting-fact question-answering task (the bAbI data) of the previous chapter; the book does not repeat any download instructions here, so the setup cell below, which is **not part of the book's code**, downloads this file from the book's public repository. Without this file the cell stops with a `FileNotFoundError`, and the remaining cells of this section cannot run, because they need `vocab` and `data`.

**Setup (not from the book).** The next cell downloads the bAbI file used by the code that follows.

In [35]:
# Setup (not from the book): download the bAbI file
!wget -q -nc -P tasksv11/en https://raw.githubusercontent.com/iamtrask/Grokking-Deep-Learning/master/tasksv11/en/qa1_single-supporting-fact_train.txt

In [36]:
import sys,random,math
from collections import Counter
import numpy as np

f = open('tasksv11/en/qa1_single-supporting-fact_train.txt','r')
raw = f.readlines()
f.close()

tokens = list()
for line in raw[0:1000]:
    tokens.append(line.lower().replace("\n","").split(" ")[1:])

new_tokens = list()
for line in tokens:
    new_tokens.append(['-'] * (6 - len(line)) + line)
tokens = new_tokens

vocab = set()
for sent in tokens:
    for word in sent:
        vocab.add(word)

vocab = list(vocab)

word2index = {}
for i,word in enumerate(vocab):
    word2index[word]=i

def words2indices(sentence):
    idx = list()
    for word in sentence:
        idx.append(word2index[word])
    return idx

indices = list()
for line in tokens:
    idx = list()
    for w in line:
        idx.append(word2index[w])
    indices.append(idx)

data = np.array(indices)

### **17.3 Training the Network**

The recurrent layer is initialized together with an embedding input. The embedding layer has `dim=16`; the `RNNCell` has `n_inputs=16`, `n_hidden=16`, and `n_output=len(vocab)`; the loss is `CrossEntropyLoss()`; the parameters of the model and of the embedding are joined in one list; and `SGD` uses `alpha=0.05`. The network is slightly more complex than the one of the previous chapter (it has one extra layer), yet the code is much simpler thanks to the little framework. According to the book, *cell* is the conventional name for a recurrent layer that implements only a single recurrence; a layer that configures an arbitrary number of cells together would be called an RNN and would take `n_layers` as an input parameter.

In [37]:
embed = Embedding(vocab_size=len(vocab),dim=16)
model = RNNCell(n_inputs=16, n_hidden=16, n_output=len(vocab))

criterion = CrossEntropyLoss()
params = model.get_parameters() + embed.get_parameters()
optim = SGD(parameters=params, alpha=0.05)

The training loop runs 1000 iterations on the first 100 examples (`batch_size = 100`). For each batch it creates the initial hidden state, feeds the first five words of each example one by one (`for t in range(5)`) through the embedding and the `RNNCell`, and uses the sixth word as the target for the final output. It then calls `loss.backward()` (without an explicit gradient, thanks to the convenience of section 13), takes an `optim.step()`, and every 200 iterations prints the loss and the fraction of correct predictions. The printed loss is the loss of the single batch divided by `len(data)/batch_size`.

The book prints these five lines:

| Iteration | Loss | % Correct |
|---|---|---|
| 0 | 0.47631100976371393 | 0.01 |
| 200 | 0.17189538896184856 | 0.28 |
| 400 | 0.1460940222788725 | 0.37 |
| 600 | 0.13845863915406884 | 0.37 |
| 800 | 0.135574472565278 | 0.37 |

In [38]:
for iter in range(1000):
    batch_size = 100
    total_loss = 0

    hidden = model.init_hidden(batch_size=batch_size)

    for t in range(5):
        input = Tensor(data[0:batch_size,t], autograd=True)
        rnn_input = embed.forward(input=input)
        output, hidden = model.forward(input=rnn_input, hidden=hidden)

    target = Tensor(data[0:batch_size,t+1], autograd=True)
    loss = criterion.forward(output, target)
    loss.backward()
    optim.step()
    total_loss += loss.data
    if(iter % 200 == 0):
        p_correct = (target.data == np.argmax(output.data,axis=1)).mean()
        print_loss = total_loss / (len(data)/batch_size)
        print("Loss:",print_loss,"% Correct:",p_correct)

Loss: 0.4632669330495068 % Correct: 0.0
Loss: 0.1735126127022746 % Correct: 0.24
Loss: 0.1564036183892806 % Correct: 0.33
Loss: 0.14375566301992107 % Correct: 0.36
Loss: 0.14445696531333965 % Correct: 0.35


Finally, the trained network is used for a single example (`batch_size = 1`): the first five words of the first training example are fed in, and the predicted word is shown together with the context. The book prints `Context: - mary moved to the` and `Pred: office.`. The network thus learns to predict the first 100 training examples with an accuracy of about 37%, which the author calls near perfect for this toy task, and it predicts a plausible place for Mary to be moving toward, much like at the end of Chapter 12.

In [39]:
batch_size = 1
hidden = model.init_hidden(batch_size=batch_size)
for t in range(5):
    input = Tensor(data[0:batch_size,t], autograd=True)
    rnn_input = embed.forward(input=input)
    output, hidden = model.forward(input=rnn_input, hidden=hidden)

target = Tensor(data[0:batch_size,t+1], autograd=True)
loss = criterion.forward(output, target)

ctx = ""
for idx in data[0:batch_size][0][0:-1]:
    ctx += vocab[idx] + " "
print("Context:",ctx)
print("Pred:", vocab[output.data.argmax()])

Context: - mary moved to the 
Pred: hallway.


---
## **18. Summary**

- **Frameworks are efficient, convenient abstractions of forward and backward logic.** They make code more readable, faster to write, faster to execute (through built-in optimizations), and much less buggy.
- A **tensor** generalizes vectors and matrices; the framework's `Tensor` class stores its numbers in a NumPy array and supports operations such as addition, negation, subtraction, multiplication, `sum`, `expand`, `transpose`, and matrix multiplication (`mm`).
- **Autograd** records a **dynamic computation graph** while forward propagation runs: each new tensor remembers its `creators` and its `creation_op`, and `.backward()` sends gradients recursively through this graph. The book keeps the dynamic approach, whereas older frameworks use static graphs.
- A tensor used more than once needs **gradient accumulation**. The `children` counters, `all_children_grads_accounted_for()`, and the `grad_origin` argument make a tensor wait until all of its children have delivered their gradients, and prevent backpropagating twice from the same child.
- Each supported function needs only two pieces: the forward method that creates a node in the graph, and its derivative as a branch of `backward`. Addition, negation, subtraction, multiplication, `sum`/`expand`, `transpose`, `mm`, `sigmoid`, `tanh`, `index_select`, and `cross_entropy` were added this way.
- With autograd, a training loop contains only **predict, compare, and learn** (`loss.backward()` followed by an update). The update is packaged in an **`SGD` optimizer** with `step()` and `zero()`.
- **Layers** (`Layer`, `Linear`, `Sequential`, `Tanh`, `Sigmoid`, `Embedding`, `RNNCell`) package forward propagation behind `.forward()` and expose their tensors through `get_parameters()`, so layers can contain other layers; **loss layers** such as `MSELoss` and `CrossEntropyLoss` have no weights.
- The **embedding layer** relies on indexing (`index_select`), whose backward pass adds each gradient row into the row that was selected. The **cross-entropy** loss contains the softmax, because computing both together is faster.
- The `RNNCell` composes three linear layers and a nonlinearity, and together with `Embedding` and `CrossEntropyLoss` it re-solves the previous chapter's task with far less code (about 37% correct on the first 100 examples).
- Viewing a framework as **autograd plus a list of prebuilt layers and optimizers** makes it easier to learn any of them; the one most similar to the framework built here is PyTorch, and it is recommended as the next step.